# Credit Card Default: Individual Fairness Preprocessing

Author: Ilse Harmers \
Last modified: March 9, 2026

Certain sections of code in this notebook have been kindly provided by dr. Mina Alishahi.

In [ ]:
# Importing libraries.
import numpy as np
import pandas as pd
pd.set_option('display.max_columns', None)
from sklearn import preprocessing
from sklearn.model_selection import train_test_split
from snsynth import Synthesizer
import snsynth.transform as tf
import utils

from sklearn.neighbors import NearestNeighbors
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
from sklearn.cluster import KMeans
from sklearn.metrics import pairwise_distances

In [ ]:
# Initializing our random forest classifier.
rf_clf = RandomForestClassifier()

def encode_ord(sample):
    """This function encodes a Credit-based synthetic dataset such that all ordinal columns are scaled to a range of (0, 1)."""

    sample["SEX"] = ((sample["SEX"] - sample["SEX"].min()) / (sample["SEX"].max() - sample["SEX"].min()))
    
    for p in [0, 2, 3, 4, 5, 6]:
        sample[f"PAY_{p}"] = ((sample[f"PAY_{p}"] - sample[f"PAY_{p}"].min()) / 
                              (sample[f"PAY_{p}"].max() - sample[f"PAY_{p}"].min()))

    return sample

def compute_similarity_fairness(indices, predictions, distances, neighbors):
    """
    Compute individual fairness for a chunk of data using precomputed k-NN distances and neighbors.

    Parameters:
        indices (list): Indices of the chunk.
        predictions (array-like): Model predictions for individuals.
        distances (array-like): Distances from k-NN.
        neighbors (array-like): Indices of k-NN neighbors.

    Returns:
        float: Mean individual fairness score for the chunk.

    Code provided by: dr. Mina Alishahi
    """
    
    fairness_scores = []
    for idx in indices:
        # Iterate through the k-nearest neighbors (skip the first as it's the individual itself).
        for neighbor_idx, distance in zip(neighbors[idx][1:], distances[idx][1:]):
            if distance < 1e-6:  # Treat very small distances as non-zero.
                distance = 1e-6
            pred_diff = abs(predictions[idx] - predictions[neighbor_idx])
            fairness_scores.append(pred_diff * distance)
    return np.mean(fairness_scores) if fairness_scores else 0.0

def similarity_fairness(predictions, features, similarity_metric = "euclidean", k = 100, num_chunks = 8):
    """
    Compute individual fairness for model predictions using k-NN.

    Parameters:
        predictions (array-like): Model predictions for individuals.
        features (array-like): Feature matrix (N x D).
        similarity_metric (str): Similarity metric ('cosine' or 'euclidean').
        k (int): Number of nearest neighbors to consider.
        num_chunks (int): Number of chunks.

    Returns:
        float: Average individual fairness score (lower is better).

    Code provided by: dr. Mina Alishahi
    """
    
    n = len(features)
    if k >= n:
        raise ValueError(f"Invalid k: {k}. Must be less than the number of data points: {n}.")
    
    # Fit k-NN on the features.
    np.random.seed(42)   # setting random seed
    knn = NearestNeighbors(n_neighbors = k + 1, metric = similarity_metric).fit(features)
    distances, neighbors = knn.kneighbors(features, return_distance=True)

    # Divide indices into chunks.
    chunk_size = max(1, n // num_chunks)
    fairness_scores = [
        compute_similarity_fairness(
            range(start, min(start + chunk_size, n)), predictions, distances, neighbors
        )
        for start in range(0, n, chunk_size)
    ]

    # Gather results from all chunks and compute the overall mean.
    return np.mean(fairness_scores) if fairness_scores else 0.0


def compute_neighborhood_consistency_fairness(indices, predictions, distances, neighbors):
    """
    Compute neighborhood consistency for a chunk of data.

    Parameters:
        indices (list): Indices of the chunk.
        predictions (array-like): Model predictions for individuals.
        distances (array-like): Distances from k-NN.
        neighbors (array-like): Indices of k-NN neighbors.

    Returns:
        float: Mean neighborhood consistency score for the chunk.

    Code provided by: dr. Mina Alishahi
    """
    consistency_scores = []
    for idx in indices:
        # Calculate consistency score for the current individual.
        local_consistency = np.mean([
            abs(predictions[idx] - predictions[neighbor_idx])
            for neighbor_idx in neighbors[idx][1:]  # Skip the first neighbor (the individual itself).
        ])
        consistency_scores.append(local_consistency)

    return np.mean(consistency_scores)

def neighborhood_consistency_fairness(predictions, features, similarity_metric = "euclidean", k = 100, num_chunks = 8):
    """
    Compute neighborhood consistency metric using k-NN.

    Parameters:
        predictions (array-like): Model predictions for individuals.
        features (array-like): Feature matrix (N x D).
        similarity_metric (str): Similarity metric ('cosine' or 'euclidean').
        k (int): Number of nearest neighbors to consider.
        num_chunks (int): Number of chunks.

    Returns:
        float: Average neighborhood consistency score (lower is better).

    Code provided by: dr. Mina Alishahi
    """
        
    # Fit k-NN on the features.
    np.random.seed(42)   # setting random seed
    knn = NearestNeighbors(n_neighbors = k + 1, metric = similarity_metric).fit(features)
    distances, neighbors = knn.kneighbors(features, return_distance = True)

    # Divide indices into chunks.
    n = len(features)
    chunk_size = n // num_chunks
    consistency_scores = [
        compute_neighborhood_consistency_fairness(
            range(start, min(start + chunk_size, n)), predictions, distances, neighbors
        )
        for start in range(0, n, chunk_size)
    ]
    
    # Gather results from all chunks and compute the overall mean.
    return np.mean(consistency_scores)

def fairness_analysis(train_set, test_set, y):
    """This function computes the similarity fairness and neigborhood consistency fairness scores for a given train and test set.
    
    train_set [array-like]: train set 
    test_set [array-like]: test set
    y [string]: column name of target attribute
    """

    np.random.seed(42)   # setting random seed
    rf_model = rf_clf.fit(train_set.drop(columns = [y]), train_set[y])
    soft_ypred = rf_model.predict_proba(test_set.drop(columns = [y]))
    ypred = rf_model.predict(test_set.drop(columns = [y]))

    # Min-max scaling the ordinal columns.
    test_copy = test_set.copy()
    test_enc = encode_ord(test_copy)
        
    sf_score = similarity_fairness(soft_ypred, test_enc.drop(columns = [y]).values)

    ncf_score = neighborhood_consistency_fairness(ypred, test_enc.drop(columns = [y]).values)

    return (sf_score, ncf_score)

def credit_encode(df):
    """This function encodes a Credit-based synthetic dataset such that all categorical columns are (one-hot) encoded and all numerical columns are
    scaled to a range of (0, 1).
    
    df [pandas DataFrame]: synthetic dataset
    """

    # One-hot encoding the "EDUCATION" and "MARRIAGE" columns in the Credit-based dataset. 
    cat_columns = ["EDUCATION", "MARRIAGE"]
    cat_encoded = utils.one_hot_encode(df[cat_columns].astype(str))

    # Transforming the continuous columns with the TableTransformer function.
    tt = tf.TableTransformer([
        tf.ChainTransformer([
            tf.LogTransformer(),
            tf.MinMaxTransformer(lower = np.log(df["LIMIT_BAL"].min()), 
                                 upper = np.log(df["LIMIT_BAL"].max()),
                                 negative = False) # LIMIT_BAL; scaling to range (0, 1)
        ]),
        tf.MinMaxTransformer(lower = df["AGE"].min(), 
                             upper = df["AGE"].max(),
                             negative = False), # AGE; scaling to range (0, 1)
        tf.ChainTransformer([
            tf.LogModulusTransformer(),
            tf.MinMaxTransformer(lower = -1 * (np.log(abs(df["BILL_AMT1"].min()) + 1)), 
                                 upper = np.log(df["BILL_AMT1"].max() + 1), 
                                 negative = False) # BILL_AMT1; scaling to range (0, 1)
        ]),
        tf.ChainTransformer([
            tf.LogModulusTransformer(),
            tf.MinMaxTransformer(lower = -1 * (np.log(abs(df["BILL_AMT2"].min()) + 1)), 
                                 upper = np.log(df["BILL_AMT2"].max() + 1), 
                                 negative = False) # BILL_AMT2; scaling to range (0, 1)
        ]),
        tf.ChainTransformer([
            tf.LogModulusTransformer(),
            tf.MinMaxTransformer(lower = -1 * (np.log(abs(df["BILL_AMT3"].min()) + 1)), 
                                 upper = np.log(df["BILL_AMT3"].max() + 1),
                                 negative = False) # BILL_AMT3; scaling to range (0, 1)
        ]),
        tf.ChainTransformer([
            tf.LogModulusTransformer(),
            tf.MinMaxTransformer(lower = -1 * (np.log(abs(df["BILL_AMT4"].min()) + 1)), 
                                 upper = np.log(df["BILL_AMT4"].max() + 1), 
                                 negative = False) # BILL_AMT4; scaling to range (0, 1)
        ]),
        tf.ChainTransformer([
            tf.LogModulusTransformer(),
            tf.MinMaxTransformer(lower = -1 * (np.log(abs(df["BILL_AMT5"].min()) + 1)), 
                                 upper = np.log(df["BILL_AMT5"].max() + 1),
                                 negative = False) # BILL_AMT5; scaling to range (0, 1)
        ]),
        tf.ChainTransformer([
            tf.LogModulusTransformer(),
            tf.MinMaxTransformer(lower = -1 * (np.log(abs(df["BILL_AMT6"].min()) + 1)), 
                                 upper = np.log(df["BILL_AMT6"].max() + 1), 
                                 negative = False) # BILL_AMT6; scaling to range (0, 1)
        ]),
        tf.ChainTransformer([
            tf.LogModulusTransformer(),
            tf.MinMaxTransformer(lower = 0, upper = np.log(df["PAY_AMT1"].max() + 1), 
                                 negative = False) # PAY_AMT1; scaling to range (0, 1)
        ]),
        tf.ChainTransformer([
            tf.LogModulusTransformer(),
            tf.MinMaxTransformer(lower = 0, upper = np.log(df["PAY_AMT2"].max() + 1), 
                                 negative = False) # PAY_AMT2; scaling to range (0, 1)
        ]),
        tf.ChainTransformer([
            tf.LogModulusTransformer(),
            tf.MinMaxTransformer(lower = 0, upper = np.log(df["PAY_AMT3"].max() + 1), 
                                 negative = False) # PAY_AMT3; scaling to range (0, 1)
        ]),
        tf.ChainTransformer([
            tf.LogModulusTransformer(),
            tf.MinMaxTransformer(lower = 0, upper = np.log(df["PAY_AMT4"].max() + 1), 
                                 negative = False) # PAY_AMT4; scaling to range (0, 1)
        ]),
        tf.ChainTransformer([
            tf.LogModulusTransformer(),
            tf.MinMaxTransformer(lower = 0, upper = np.log(df["PAY_AMT5"].max() + 1), 
                                 negative = False) # PAY_AMT5; scaling to range (0, 1)
        ]),
        tf.ChainTransformer([
            tf.LogModulusTransformer(),
            tf.MinMaxTransformer(lower = 0, upper = np.log(df["PAY_AMT6"].max() + 1), 
                                 negative = False) # PAY_AMT6; scaling to range (0, 1)
        ]),
    ])

    tf_cont_cols = np.array(synth._get_train_data(
                        df[['LIMIT_BAL', 'AGE', 'BILL_AMT1', 'BILL_AMT2', 'BILL_AMT3', 'BILL_AMT4', 'BILL_AMT5', 
                            'BILL_AMT6', 'PAY_AMT1', 'PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4','PAY_AMT5', 'PAY_AMT6']],
                        transformer = tt, preprocessor_eps = 0.0, style = 'gan', nullable = False, categorical_columns=None,
                        ordinal_columns=None, continuous_columns=None,))

    num_encoded = pd.DataFrame(tf_cont_cols, columns = ['LIMIT_BAL', 'AGE', 'BILL_AMT1', 'BILL_AMT2',
                                                        'BILL_AMT3', 'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6', 'PAY_AMT1', 
                                                        'PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6'])

    # Concatenating all the columns back together.
    train_encoded = pd.concat([num_encoded.reset_index(drop = True), cat_encoded.reset_index(drop = True),
                               df[['SEX', 'PAY_0', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6', 'DEFAULT']].reset_index(drop = True)], axis = 1)

    return train_encoded

## Pre-Processing Credit Card Default Dataset

In [ ]:
# Importing Credit Card Default dataset (downloaded from https://archive.ics.uci.edu/dataset/350/default+of+credit+card+clients on May 28, 2025).

# Assigning file path for importing the dataset. 
path = "../credit-card-default/credit-card-clients.xls"

# Importing dataset as pandas DataFrame.
credit = pd.read_excel(path, index_col = 0, header = 1)
credit = credit.rename(columns = {"default payment next month": "DEFAULT"})
dataset_name = "Credit"
column_names = list(credit.columns)
credit

In [ ]:
# Print names of columns in the dataset as a check.
print(f"{dataset_name} contains {len(column_names)} columns: \n {column_names} \n")

# Check dtypes of dataset's columns.
print("Columns' dtypes:\n", credit.dtypes)
print()

# Count missing values (reported as NaNs) in each column.
missing_values = np.array([credit[col].isna().sum() for col in column_names])

# This if-elif statement checks whether the columns in the dataset contain missing values that are reported as NaNs. 
if missing_values.any() == False:   # no non-zero entries
    print(f"{dataset_name} has no missing values in any of its columns.")
elif missing_values.any() == True:   # at least one non-zero entry (indicating the presence of missing values)
    print(f"There are {np.sum(missing_values)} missing values in {dataset_name}'s columns.")   
    for i in range(len(missing_values)):
        print(f"\t{column_names[i]}: {missing_values[i]} values")

In [ ]:
# Statistical summary of the dataset.
credit.describe()

In [ ]:
print("Number of rows with unusual labels: ", 
      credit.loc[(credit["EDUCATION"] == 0) | (credit["EDUCATION"] == 5) | (credit["EDUCATION"] == 6) | (credit["MARRIAGE"] == 0)].shape[0])

In [ ]:
credit = credit.drop(index = credit.loc[(credit["EDUCATION"] == 0) | (credit["EDUCATION"] == 5) | (credit["EDUCATION"] == 6) | 
                                        (credit["MARRIAGE"] == 0)].index)

In [ ]:
# Statistical summary of the dataset.
credit.describe()

## Individual Fairness Preprocessing

In [ ]:
# Initializing the synthesizer for transforming the Credit dataset with the TableTransformer function. 
synth = Synthesizer.create('pategan', epsilon = 10.0, delta = 1e-05, plot_losses = True)

credit_train, credit_test = train_test_split(credit, train_size = 0.80, random_state = 42)
train_encoded = credit_encode(credit_train)
test_encoded = credit_encode(credit_test)

In [ ]:
test_encoded.describe()#, train_encoded.describe()

### "No Pre-Processing" Reference

In [ ]:
# Individual fairness metrics.
credit_sf, credit_ncf = fairness_analysis(train_set = train_encoded, test_set = test_encoded, y = "DEFAULT")
print("sf_score: ", credit_sf)  
print("ncf_score: ", credit_ncf)

# Encoding the sensitive attribute for the fairness analysis. Note regarding 'SEX': 1 = male; 2 = female.
sex_encoded = utils.one_hot_encode(credit_train[["SEX"]], order = [[2, 1]])
cols_encoded = pd.concat([sex_encoded.reset_index(drop = True), credit_train["DEFAULT"].reset_index(drop = True)], axis = 1)
print("Demographic parity:", utils.demographic_parity(df = cols_encoded, s = "SEX", y = "DEFAULT"))
print("Disparate impact:", utils.disparate_impact(df = cols_encoded, s = "SEX", y = "DEFAULT"))

np.random.seed(42)
rf = rf_clf.fit(train_encoded.drop(columns = ["DEFAULT"]), train_encoded["DEFAULT"])
ypred = rf.predict(test_encoded.drop(columns = ["DEFAULT"]))
auroc = roc_auc_score(test_encoded["DEFAULT"], ypred)
print(f"AUROC on test data: {auroc:.8f}")

## Train KMeans Clustering Algorithm

In [ ]:
# Min-max scaling the ordinal columns.
train_ord = train_encoded.copy()
train_nom = encode_ord(train_ord)
test_ord = test_encoded.copy()
test_nom = encode_ord(test_ord)

In [ ]:
clusters = np.arange(50, 600, 50)
clusters_test = np.arange(50, 600, 50)
thres = 0.5 
for clus in range(len(clusters)):
    # Training the KMeans clustering algorithm on the min-max scaled train set.
    print(f"Cluster size (train) = {clusters[clus]}")
    kmeans = KMeans(n_clusters = clusters[clus], random_state = 0, n_init = "auto").fit(train_nom.drop(columns = ["DEFAULT"]))
    pred = kmeans.predict(train_nom.drop(columns = ["DEFAULT"]))
    print("kmeans inertia:", kmeans.inertia_)

    train_kmeans = train_nom.copy()
    train_kmeans["cluster"] = pred

    # If an entry in the train set has a distance to its assigned cluster's center smaller than the threshold, the entry's target label is 
    # re-assigned to the majority label of the cluster (if this is not the case already).
    distances = [pairwise_distances(
                    np.array([kmeans.cluster_centers_[c]]), train_kmeans[train_kmeans["cluster"] == c].drop(columns = ["DEFAULT", "cluster"]).values
                    )[0] for c in np.unique(pred)]
    for c in range(len(np.unique(pred))):
        index = train_kmeans[train_kmeans["cluster"] == np.unique(pred)[c]].iloc[np.where(distances[c] <= thres)].index
        train_kmeans.loc[index, "DEFAULT"] = train_kmeans[train_kmeans["cluster"] == np.unique(pred)[c]]["DEFAULT"].value_counts().index[0]

    print("KMeans (DEFAULT = 1):", train_kmeans[train_kmeans["DEFAULT"] == 1].shape)
    print("Original train set (DEFAULT = 1):", train_encoded[train_encoded["DEFAULT"] == 1].shape)

    # Training the KMeans clustering algorithm on the min-max scaled test set.
    print(f"Cluster size (test) = {clusters_test[clus]}")
    kmeans = KMeans(n_clusters = clusters_test[clus], random_state = 3, n_init = "auto").fit(test_nom.drop(columns = ["DEFAULT"]))
    pred_test = kmeans.predict(test_nom.drop(columns = ["DEFAULT"]))

    test_kmeans = test_nom.copy()
    test_kmeans["cluster"] = pred_test

    # If an entry in the test set has a distance to its assigned cluster's center smaller than the threshold, the entry's target label is 
    # re-assigned to the majority label of the cluster (if this is not the case already).
    distances = [pairwise_distances(
                    np.array([kmeans.cluster_centers_[c]]), test_kmeans[test_kmeans["cluster"] == c].drop(columns = ["DEFAULT", "cluster"]).values
                    )[0] for c in np.unique(pred_test)]
    for c in range(len(np.unique(pred_test))):
        index = test_kmeans[test_kmeans["cluster"] == np.unique(pred_test)[c]].iloc[np.where(distances[c] <= thres)].index
        test_kmeans.loc[index, "DEFAULT"] = test_kmeans[test_kmeans["cluster"] == np.unique(pred_test)[c]]["DEFAULT"].value_counts().index[0]

    print("KMeans (DEFAULT = 1):", test_kmeans[test_kmeans["DEFAULT"] == 1].shape)
    print("Original test set (DEFAULT = 1):", test_encoded[test_encoded["DEFAULT"] == 1].shape)

    ctrain = train_encoded.copy()
    ctrain["DEFAULT"] = train_kmeans["DEFAULT"]
    ctest = test_encoded.copy()
    ctest["DEFAULT"] = test_kmeans["DEFAULT"]

    credit_sf, credit_ncf = fairness_analysis(train_set = ctrain, test_set = ctest, y = "DEFAULT")
    print("sf_score: ", credit_sf)  
    print("ncf_score: ", credit_ncf)

    np.random.seed(42)
    rf = rf_clf.fit(ctrain.drop(columns = ["DEFAULT"]), ctrain["DEFAULT"])
    ypred = rf.predict(ctest.drop(columns = ["DEFAULT"]))
    auroc = roc_auc_score(ctest["DEFAULT"], ypred)
    print(f"AUROC on test data: {auroc:.8f}")
    print()

In [ ]:
clusters = [250] * 11
clusters_test = np.arange(50, 600, 50)
thres = 0.5 
for clus in range(len(clusters)):
    # Training the KMeans clustering algorithm on the min-max scaled train set.
    print(f"Cluster size (train) = {clusters[clus]}")
    kmeans = KMeans(n_clusters = clusters[clus], random_state = 0, n_init = "auto").fit(train_nom.drop(columns = ["DEFAULT"]))
    pred = kmeans.predict(train_nom.drop(columns = ["DEFAULT"]))
    print("kmeans inertia:", kmeans.inertia_)

    train_kmeans = train_nom.copy()
    train_kmeans["cluster"] = pred

    # If an entry in the train set has a distance to its assigned cluster's center smaller than the threshold, the entry's target label is 
    # re-assigned to the majority label of the cluster (if this is not the case already).
    distances = [pairwise_distances(
                    np.array([kmeans.cluster_centers_[c]]), train_kmeans[train_kmeans["cluster"] == c].drop(columns = ["DEFAULT", "cluster"]).values
                    )[0] for c in np.unique(pred)]
    for c in range(len(np.unique(pred))):
        index = train_kmeans[train_kmeans["cluster"] == np.unique(pred)[c]].iloc[np.where(distances[c] <= thres)].index
        train_kmeans.loc[index, "DEFAULT"] = train_kmeans[train_kmeans["cluster"] == np.unique(pred)[c]]["DEFAULT"].value_counts().index[0]

    print("KMeans (DEFAULT = 1):", train_kmeans[train_kmeans["DEFAULT"] == 1].shape)
    print("Original train set (DEFAULT = 1):", train_encoded[train_encoded["DEFAULT"] == 1].shape)

    # Training the KMeans clustering algorithm on the min-max scaled test set.
    print(f"Cluster size (test) = {clusters_test[clus]}")
    kmeans = KMeans(n_clusters = clusters_test[clus], random_state = 3, n_init = "auto").fit(test_nom.drop(columns = ["DEFAULT"]))
    pred_test = kmeans.predict(test_nom.drop(columns = ["DEFAULT"]))

    test_kmeans = test_nom.copy()
    test_kmeans["cluster"] = pred_test

    # If an entry in the test set has a distance to its assigned cluster's center smaller than the threshold, the entry's target label is 
    # re-assigned to the majority label of the cluster (if this is not the case already).
    distances = [pairwise_distances(
                    np.array([kmeans.cluster_centers_[c]]), test_kmeans[test_kmeans["cluster"] == c].drop(columns = ["DEFAULT", "cluster"]).values
                    )[0] for c in np.unique(pred_test)]
    for c in range(len(np.unique(pred_test))):
        index = test_kmeans[test_kmeans["cluster"] == np.unique(pred_test)[c]].iloc[np.where(distances[c] <= thres)].index
        test_kmeans.loc[index, "DEFAULT"] = test_kmeans[test_kmeans["cluster"] == np.unique(pred_test)[c]]["DEFAULT"].value_counts().index[0]

    print("KMeans (DEFAULT = 1):", test_kmeans[test_kmeans["DEFAULT"] == 1].shape)
    print("Original test set (DEFAULT = 1):", test_encoded[test_encoded["DEFAULT"] == 1].shape)

    ctrain = train_encoded.copy()
    ctrain["DEFAULT"] = train_kmeans["DEFAULT"]
    ctest = test_encoded.copy()
    ctest["DEFAULT"] = test_kmeans["DEFAULT"]

    credit_sf, credit_ncf = fairness_analysis(train_set = ctrain, test_set = ctest, y = "DEFAULT")
    print("sf_score: ", credit_sf)  
    print("ncf_score: ", credit_ncf)

    np.random.seed(42)
    rf = rf_clf.fit(ctrain.drop(columns = ["DEFAULT"]), ctrain["DEFAULT"])
    ypred = rf.predict(ctest.drop(columns = ["DEFAULT"]))
    auroc = roc_auc_score(ctest["DEFAULT"], ypred)
    print(f"AUROC on test data: {auroc:.8f}")
    print()

In [ ]:
clus_train = 250
clus_test = 450
thres = 0.5

# Training the KMeans clustering algorithm on the min-max scaled train set.
print(f"Cluster size (train) = {clus_train}")
kmeans = KMeans(n_clusters = clus_train, random_state = 0, n_init = "auto").fit(train_nom.drop(columns = ["DEFAULT"]))
pred = kmeans.predict(train_nom.drop(columns = ["DEFAULT"]))
print("kmeans inertia:", kmeans.inertia_)

train_kmeans = train_nom.copy()
train_kmeans["cluster"] = pred

# If an entry in the train set has a distance to its assigned cluster's center smaller than the threshold, the entry's target label is 
# re-assigned to the majority label of the cluster (if this is not the case already).
distances = [pairwise_distances(
                np.array([kmeans.cluster_centers_[c]]), train_kmeans[train_kmeans["cluster"] == c].drop(columns = ["DEFAULT", "cluster"]).values
                )[0] for c in np.unique(pred)]
for c in range(len(np.unique(pred))):
    index = train_kmeans[train_kmeans["cluster"] == np.unique(pred)[c]].iloc[np.where(distances[c] <= thres)].index
    train_kmeans.loc[index, "DEFAULT"] = train_kmeans[train_kmeans["cluster"] == np.unique(pred)[c]]["DEFAULT"].value_counts().index[0]

print("Train set (DEFAULT = 1):", credit_train[credit_train["DEFAULT"] == 1].shape)
print("KMeans (DEFAULT = 1):", train_kmeans[train_kmeans["DEFAULT"] == 1].shape)
new_train = credit_train.copy()
new_train["DEFAULT"] = train_kmeans["DEFAULT"].values
print("New train set (DEFAULT = 1):", new_train[new_train["DEFAULT"] == 1].shape)
new_train.to_csv("train-test-datasets/credit-card-default/credit_train.csv", index = False)
print()

# Training the KMeans clustering algorithm on the min-max scaled test set.
print(f"Cluster size (test) = {clus_test}")
kmeans = KMeans(n_clusters = clus_test, random_state = 3, n_init = "auto").fit(test_nom.drop(columns = ["DEFAULT"]))
pred_test = kmeans.predict(test_nom.drop(columns = ["DEFAULT"]))

test_kmeans = test_nom.copy()
test_kmeans["cluster"] = pred_test

# If an entry in the test set has a distance to its assigned cluster's center smaller than the threshold, the entry's target label is 
# re-assigned to the majority label of the cluster (if this is not the case already).
distances = [pairwise_distances(
                np.array([kmeans.cluster_centers_[c]]), test_kmeans[test_kmeans["cluster"] == c].drop(columns = ["DEFAULT", "cluster"]).values
                )[0] for c in np.unique(pred_test)]
for c in range(len(np.unique(pred_test))):
    index = test_kmeans[test_kmeans["cluster"] == np.unique(pred_test)[c]].iloc[np.where(distances[c] <= thres)].index
    test_kmeans.loc[index, "DEFAULT"] = test_kmeans[test_kmeans["cluster"] == np.unique(pred_test)[c]]["DEFAULT"].value_counts().index[0]

print("Test set (DEFAULT = 1):", credit_test[credit_test["DEFAULT"] == 1].shape)
print("KMeans (DEFAULT = 1):", test_kmeans[test_kmeans["DEFAULT"] == 1].shape)
new_test = credit_test.copy()
new_test["DEFAULT"] = test_kmeans["DEFAULT"].values
print("New test set (DEFAULT = 1):", new_test[new_test["DEFAULT"] == 1].shape)
new_test.to_csv("train-test-datasets/credit-card-default/credit_test.csv", index = False)
print()

ntrain_encoded = credit_encode(new_train)
ntest_encoded = credit_encode(new_test)
# Checking for missing columns between the train and test set. 
missing_cols = list(set(list(ntrain_encoded.columns)) - set(list(ntest_encoded.columns)))
print("Missing column(s):", missing_cols)
# If there is a missing column, most likely due to a label from the train set not being present in the test set, 
# then we reinsert a 'zero' column into the test set at the right place to account for this missing label.
if missing_cols:
    for c in missing_cols:
        df_col = pd.DataFrame({c: ntest_encoded["AGE"]*0})
        ntest_encoded.insert(0, c, value = df_col)
ntest_encoded = ntest_encoded[ntrain_encoded.columns]

credit_sf, credit_ncf = fairness_analysis(train_set = ntrain_encoded, test_set = ntest_encoded, y = "DEFAULT")
print("sf_score: ", credit_sf)  
print("ncf_score: ", credit_ncf)

np.random.seed(42)
rf = rf_clf.fit(ntrain_encoded.drop(columns = ["DEFAULT"]), ntrain_encoded["DEFAULT"])
ypred = rf.predict(ntest_encoded.drop(columns = ["DEFAULT"]))
auroc = roc_auc_score(ntest_encoded["DEFAULT"], ypred)
print(f"AUROC on test data: {auroc:.8f}")